In [ ]:
#notebook to do baseline comparisons for different models on a dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications
!pip install -e ".[experiment]"

Mounted at /content/drive
/content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications
Obtaining file:///content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 120.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.1 MB/s eta 0:00:00

In [2]:
!pip install .

Processing /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformercompression: filename=transformercompression-0.0.1-py3-none-any.whl size=38585 sha256=01044fd829bdd353e7342a3f32439b4d773ca52e0a66c2dd3bec2b5758ce3a17
  Stored in directory: /tmp/pip-ephem-wheel-cache-eic5ph1r/wheels/f3/6c/2c/64e357a17a227b3c2a1e0484935ae90ce5b5ea18616539f7ae
Successfully built transformercompression
  Attempting uninstall: transformercompression
    Found existing installation: transformercompression 0.0.1
    Uninstalling transformercompression-0.0.1:
      Successfully uninstalled transformercompression-0.0.1


In [3]:
!pip install -q "peft==0.13.2" "transformers==4.41.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 7.6 MB/s eta 0:00:00


In [4]:
import os

BASE_RESULTS_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments"
LOG_DIR = os.path.join(BASE_RESULTS_DIR, "logs")
MODEL_DIR = os.path.join(BASE_RESULTS_DIR, "models")
BASE_EVAL_DIR = os.path.join(BASE_RESULTS_DIR, "eval")

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(BASE_EVAL_DIR, exist_ok=True)

In [5]:
import logging
import sys

# Get the root logger or a named logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)  # allow INFO and above

for h in list(logger.handlers):
  logger.removeHandler(h)

# Create a handler that writes to stdout
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)

# (Optional) set a formatting for readability
formatter = logging.Formatter('%(levelname)s - %(message)s')
handler.setFormatter(formatter)

# Add handler to the logger
logger.addHandler(handler)

# Now test
logger.info("This will be printed to stdout")
logger.debug("This will not print (level is INFO)")

INFO - This will be printed to stdout


In [6]:
from lm_eval.tasks import initialize_tasks
initialize_tasks()

INFO - NumExpr defaulting to 12 threads.
INFO - PyTorch version 2.9.0+cu126 available.
INFO - TensorFlow version 2.19.0 available.
INFO - JAX version 0.7.2 available.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


WARNING - Some tasks could not be loaded due to missing dependencies. Run with `--verbosity DEBUG` for full details.
WARNING - Some tasks could not be loaded due to missing dependencies. Run with `--verbosity DEBUG` for full details.


In [9]:
import json

import lm_eval
# import torch
from lm_eval import tasks
from lm_eval import utils as lm_eval_utils
from lm_eval.api.registry import ALL_TASKS
from lm_eval.models.huggingface import HFLM
from lm_eval.tasks import initialize_tasks

from slicegpt import gpu_utils, hf_utils, utils
from slicegpt.config import config

logging.basicConfig(
    level=logging.DEBUG,
    stream=sys.stdout,
    format="%(levelname)s: %(message)s"
)
def eval(args):
    """
    Run LM Evaluation Harness on either:
      - a sliced (pruned) model, if 'sliced_model_path' is provided
      - or a dense HF model, if not.
    """

    logger.info("Running Evaluation")

    use_sliced = args.get("sliced_model_path") is not None and args.get("sparsity", 0.0) > 0.0

    if use_sliced:
        # ---------- SLICED MODEL BRANCH ----------
        logger.info(
            f"Loading SLICED model {args['model']} from {args['sliced_model_path']} "
            f"with sparsity {args['sparsity']}"
        )

        model_adapter, tokenizer = hf_utils.load_sliced_model(
            args["model"],
            args["sliced_model_path"],
            sparsity=args["sparsity"],
            token=None,
            round_interval=args.get("round_interval", None),
        )

        # For sliced models, disable weight tying
        if hasattr(model_adapter.model, "tie_weights"):
            model_adapter.model.tie_weights = lambda *x, **kw: None

        model_adapter.model.to(config.device)

        hflm = HFLM(
            pretrained=model_adapter.model,
            tokenizer=tokenizer,
            batch_size=args["batch_size"],
        )

    else:
        # ---------- DENSE BASELINE BRANCH ----------
        logger.info(
            f"Loading DENSE baseline model {args['model']} directly from HF hub"
        )

        # Here we let LM Eval Harness load the model & tokenizer itself
        hflm = HFLM(
            pretrained=args["model"],
            batch_size=args["batch_size"],
        )

    # ---------- Task selection ----------
    if args["tasks"] is None:
        logger.warning(
            "args['tasks'] is None -> using ALL_TASKS. "
            "This may be very slow / memory-heavy."
        )
        task_names = tasks.ALL_TASKS
    else:
        if isinstance(args["tasks"], str):
            patterns = [t.strip() for t in args["tasks"].split(",") if t.strip()]
        else:
            patterns = args["tasks"]
        task_names = lm_eval_utils.pattern_match(patterns, ALL_TASKS)

    logger.info(f"Selected Tasks: {task_names}")

    # ---------- Run evaluation ----------
    results = lm_eval.simple_evaluate(
        hflm,
        tasks=task_names,
        num_fewshot=args["num_fewshot"],
        batch_size=args["batch_size"],
        limit=args["limit"],
        write_out=True,
        log_samples=False,
    )

    logger.info("Results (metrics only):")
    logger.info(results["results"])


    # ---------- Save results ----------
    os.makedirs(args["save_dir"], exist_ok=True)

    # Use sparsity=0.0 if not present, so filenames still make sense
    sparsity_val = float(args.get("sparsity", 0.0))
    sparsity_tag = f"{sparsity_val:.2f}"

    result_path = os.path.join(
        args["save_dir"],
        f"results_s{sparsity_tag}_{'_'.join(task_names)}_light.json",
    )

    with open(result_path, "w") as f:
        json.dump(results, f, indent=2)

    logger.info(f"Saved results to {result_path}")

    return results


In [10]:
args = {
    "model": "facebook/opt-125m",
    #"model": "facebook/opt-1.3b",
    "sliced_model_path": None,   # IMPORTANT
    "sparsity": 0.0,
    "round_interval": None,
    "batch_size": 64,
    "tasks": ["squadv2"],
    "num_fewshot": 0,
    "limit": None,
    "save_dir": "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/",
}

eval(args)

INFO - Running Evaluation
INFO - Loading DENSE baseline model facebook/opt-125m directly from HF hub
INFO - Using device 'cuda'


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/251M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

...

INFO - Running loglikelihood requests



100%|██████████| 11873/11873 [00:10<00:00, 1149.79it/s]
/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/squadv2/task.py:40: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  squad_metric = datasets.load_metric("squad_v2")
/usr/local/lib/python3.12/dist-packages/datasets/load.py:756: FutureWarning: The repository for squad_v2 contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.18.0/metrics/squad_v2/squad_v2.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


/usr/local/lib/python3.12/dist-packages/datasets/load.py:756: FutureWarning: The repository for squad_v2 contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.18.0/metrics/squad_v2/squad_v2.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:756: FutureWarning: The repository for squad_v2 contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.18.0/metrics/squad_v2/squad_v2.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric 

INFO - Results (metrics only):
INFO - {'squadv2': {'exact,none': 0.5390381537943233, 'exact_stderr,none': 'N/A', 'f1,none': 2.8783499249418556, 'f1_stderr,none': 'N/A', 'HasAns_exact,none': 0.8265856950067476, 'HasAns_exact_stderr,none': 'N/A', 'HasAns_f1,none': 5.511917789951863, 'HasAns_f1_stderr,none': 'N/A', 'NoAns_exact,none': 0.2523128679562658, 'NoAns_exact_stderr,none': 'N/A', 'NoAns_f1,none': 0.2523128679562658, 'NoAns_f1_stderr,none': 'N/A', 'best_exact,none': 50.07159100480081, 'best_exact_stderr,none': 'N/A', 'best_f1,none': 50.07159100480081, 'best_f1_stderr,none': 'N/A', 'alias': 'squadv2'}}
INFO - Saved results to /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/results_s0.00_squadv2_light.json


{'results': {'squadv2': {'exact,none': 0.5390381537943233,
   'exact_stderr,none': 'N/A',
   'f1,none': 2.8783499249418556,
   'f1_stderr,none': 'N/A',
   'HasAns_exact,none': 0.8265856950067476,
   'HasAns_exact_stderr,none': 'N/A',
   'HasAns_f1,none': 5.511917789951863,
   'HasAns_f1_stderr,none': 'N/A',
   'NoAns_exact,none': 0.2523128679562658,
   'NoAns_exact_stderr,none': 'N/A',
   'NoAns_f1,none': 0.2523128679562658,
   'NoAns_f1_stderr,none': 'N/A',
   'best_exact,none': 50.07159100480081,
   'best_exact_stderr,none': 'N/A',
   'best_f1,none': 50.07159100480081,
   'best_f1_stderr,none': 'N/A',
   'alias': 'squadv2'}},
 'configs': {'squadv2': {'description': '',
   'target_delimiter': ' ',
   'fewshot_delimiter': '\n\n',
   'num_fewshot': 0,
   'output_type': 'generate_until',
   'generation_kwargs': {'until': ['\n\n'], 'do_sample': False},
   'repeats': 1,
   'should_decontaminate': False}},
 'versions': {'squadv2': 3},
 'n-shot': {'squadv2': 0},
 'config': {'model': 'faceboo

In [ ]:
import json

RESULT_FILE = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense/full_results_0_shot.json"

with open(RESULT_FILE, "r") as f:
    data = json.load(f)

print("Dense model accuracy (acc,none):")
for task, metrics in data.items():
    # each `metrics` is a dict like {"acc,none": ..., "acc_stderr,none": ..., ...}
    acc = metrics.get("acc,none")
    print(f"{task}: {acc}")


Dense model accuracy (acc,none):
arc_challenge: 0.19112627986348124
arc_easy: 0.4351851851851852
hellaswag: 0.29177454690300736
piqa: 0.6300326441784548
winogrande: 0.5035516969218626


In [ ]:
for task, metrics in data.items():
    acc_norm = metrics.get("acc_norm,none")
    print(f"{task}: {acc_norm}")

arc_challenge: 0.22781569965870307
arc_easy: 0.39941077441077444
hellaswag: 0.3134833698466441
piqa: 0.6202393906420022
winogrande: None


In [ ]:
import json

SUMMARY_FILE = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense/0_shot_task_results.json"  # whatever its path is

with open(SUMMARY_FILE, "r") as f:
    summary = json.load(f)

print("Dense model normalized accuracies:")
for task, score in summary.items():
    print(f"{task}: {score}")

Dense model normalized accuracies:
arc_challenge: 0.2278
arc_easy: 0.3994
hellaswag: 0.3135
piqa: 0.6202
winogrande: 0.5036
average: 0.4129
